# fNIRS benchmark serial smoke-test notebook

This notebook isolates the first `no_hrf` job **without** `ProcessPoolExecutor` so you can find out whether the failure is in:

1. Python preprocessing / GLM,
2. MATLAB bundle construction, or
3. MATLAB Engine execution.

Run the cells top to bottom. The notebook only touches **one subject** and a couple of representative pipelines unless you expand it.


In [1]:

from pathlib import Path
import importlib.util
import traceback
import time
import json

import numpy as np
import pandas as pd
import mne

# --- EDIT THESE PATHS IF NEEDED ---
SCRIPT_PATH = Path('/home/asunkari/fnirs-representation-learning/v1/fnirs_benchmark_v7_post_changes.py')
ROOT = Path('/home/asunkari/fnirs-representation-learning')
TRUTH_TEMPLATE_DIR = ROOT / 'synthetic_hrf_generation'
ANALYZIR_PATH = Path('/home/asunkari/nirs-toolbox')
SUBJECT = 'Subj94'

assert SCRIPT_PATH.exists(), SCRIPT_PATH
assert ROOT.exists(), ROOT
assert ANALYZIR_PATH.exists(), ANALYZIR_PATH
print('Using script:', SCRIPT_PATH)
print('Using root:', ROOT)
print('Using truth dir:', TRUTH_TEMPLATE_DIR)
print('Using AnalyzIR path:', ANALYZIR_PATH)


Using script: /home/asunkari/fnirs-representation-learning/v1/fnirs_benchmark_v7_post_changes.py
Using root: /home/asunkari/fnirs-representation-learning
Using truth dir: /home/asunkari/fnirs-representation-learning/synthetic_hrf_generation
Using AnalyzIR path: /home/asunkari/nirs-toolbox


In [2]:
import sys
import importlib.util

spec = importlib.util.spec_from_file_location("bench", str(SCRIPT_PATH))
bench = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = bench
spec.loader.exec_module(bench)

print("Loaded module:", bench.__name__)
print("MATLAB engine available:", bench.optional_import_matlab_engine()[1] is not None)

Loaded module: bench
MATLAB engine available: True


In [3]:

# Build a serial config that matches your local smoke test.
config = bench.BenchmarkConfig(
    root=str(ROOT),
    truth_template_dir=str(TRUTH_TEMPLATE_DIR),
    analyzir_path=str(ANALYZIR_PATH),
    use_matlab=True,
    use_matlab_engine=True,
    prefer_matlab_engine=True,
    empirical_null_shift_count=1,
    n_workers=1,
    overwrite=True,
)
config.file_specs = bench.default_file_specs()
config.pipeline_specs = bench.default_pipeline_specs()

pd.DataFrame([bench.asdict(fs) for fs in config.file_specs])


,label,filename,amplitude_value,is_null,annotation_source_filename
0,no_hrf,resting_clean.snirf,0,True,resting_hrf_20.snirf
1,hrf_20,resting_hrf_20.snirf,20,False,NaN
2,hrf_50,resting_hrf_50.snirf,50,False,NaN
3,hrf_100,resting_hrf_100.snirf,100,False,NaN


In [4]:
df_pipes = pd.DataFrame([bench.asdict(ps) for ps in config.pipeline_specs])
df_pipes[['label','backend','nuisance_method','hrf_model','solver','pruning_style','motion_method','filter_mode','use_block_average']]

,label,backend,nuisance_method,hrf_model,solver,pruning_style,motion_method,filter_mode,use_block_average
0,NoSS_Glover_AUTO,python,none,glover,auto,strict_combined,tddr,bandpass,False
1,LocalSS_Glover_AUTO,python,local_nearest,glover,auto,strict_combined,tddr,bandpass,False
2,PooledPCA2_Glover_AUTO,python,pooled_pca2,glover,auto,strict_combined,tddr,bandpass,False
3,SSAuxPCA_Glover_AUTO,python,ss_aux_pca,glover,auto,strict_combined,tddr,bandpass,False
4,MultiSSOrth3_Glover_AUTO,python,multi_ss_orth3,glover,auto,strict_combined,tddr,bandpass,False
5,NoSS_Glover_OLS,python,none,glover,ols,strict_combined,tddr,bandpass,False
6,NoSS_Glover_ARIRLS,matlab_arirls,none,glover,arirls,strict_combined,tddr,bandpass,False
7,LocalSS_Glover_OLS,python,local_nearest,glover,ols,strict_combined,tddr,bandpass,False
8,LocalSS_Glover_ARIRLS,matlab_arirls,local_nearest,glover,arirls,strict_combined,tddr,bandpass,False
9,LocalSS_SPM_AUTO,python,local_nearest,spm,auto,strict_combined,tddr,bandpass,False


## Helpers

`prepare_job_state()` reproduces the early steps of `process_subject_file_job()` in the **main notebook process**.


In [5]:

def prepare_job_state(file_label='no_hrf'):
    file_spec = next(fs for fs in config.file_specs if fs.label == file_label)
    dataset_dir = config.dataset_path()
    subject_dir = dataset_dir / SUBJECT
    snirf_file_path = subject_dir / file_spec.filename
    annotation_source_path = subject_dir / file_spec.annotation_source_filename if file_spec.annotation_source_filename else None
    reference_path = subject_dir / 'resting_hrf_20.snirf'

    if not snirf_file_path.exists():
        raise FileNotFoundError(snirf_file_path)
    if not reference_path.exists():
        raise FileNotFoundError(reference_path)

    print(f'Loading raw CW from: {snirf_file_path}')
    raw_cw = mne.io.read_raw_snirf(snirf_file_path, preload=True, verbose=False)
    if file_spec.is_null:
        print(f'Copying annotations from: {annotation_source_path}')
        raw_cw = bench.copy_valid_annotations(raw_cw, annotation_source_path)
    raw_cw = bench.sanitize_annotations_to_single_task(raw_cw)

    print(f'Loading reference truth file: {reference_path}')
    reference_raw = mne.io.read_raw_snirf(reference_path, preload=True, verbose=False)
    data_type_labels = bench.read_measurement_data_type_labels(reference_path)
    if data_type_labels is None:
        raise RuntimeError('Could not read truth labels from reference file.')

    picks_cw = bench.get_cw_channel_indices(reference_raw)
    cw_names = np.asarray(reference_raw.ch_names)[picks_cw]
    if len(data_type_labels) == len(cw_names):
        aligned_names = cw_names
    elif len(data_type_labels) == len(reference_raw.ch_names):
        aligned_names = np.asarray(reference_raw.ch_names)
    else:
        raise RuntimeError('Truth-label alignment failed.')

    target_pair_names = sorted(set(name.split(' ')[0] for name in aligned_names[data_type_labels == 1].tolist()))
    target_pair_set = set(target_pair_names)

    cw_channel_table = bench.build_cw_channel_table(reference_raw, SUBJECT, file_spec.label, config)
    long_pair_names = sorted(cw_channel_table.loc[cw_channel_table['group'] == 'LS', 'pair_name'].astype(str).unique())
    non_target_pair_names = [pair for pair in long_pair_names if pair not in target_pair_set]

    channel_quality, pair_quality = bench.build_quality_tables(raw_cw, SUBJECT, file_spec.label, config)
    truth_templates = bench.load_truth_templates(config)

    state = {
        'file_spec': file_spec,
        'subject_dir': subject_dir,
        'snirf_file_path': snirf_file_path,
        'raw_cw': raw_cw,
        'reference_raw': reference_raw,
        'target_pair_names': target_pair_names,
        'target_pair_set': target_pair_set,
        'non_target_pair_names': non_target_pair_names,
        'cw_channel_table': cw_channel_table,
        'channel_quality': channel_quality,
        'pair_quality': pair_quality,
        'truth_templates': truth_templates,
    }
    return state


In [6]:

state = prepare_job_state('no_hrf')
print('Annotations:', len(state['raw_cw'].annotations))
print('True target pairs:', state['target_pair_names'])
print('n true targets:', len(state['target_pair_names']))
print('n true non-target pairs:', len(state['non_target_pair_names']))

state['pair_quality'].head()


Loading raw CW from: /home/asunkari/fnirs-representation-learning/snirf_dataset_2/Subj94/resting_clean.snirf


/tmp/ipykernel_574312/389262916.py:15: RuntimeWarning: The data only contains 2D location information for the optode positions. It is highly recommended that data is used which contains 3D location information for the optode positions. With only 2D locations it can not be guaranteed that MNE functions will behave correctly and produce accurate results. If it is not possible to include 3D positions in your data, please consider using the set_montage() function.
  raw_cw = mne.io.read_raw_snirf(snirf_file_path, preload=True, verbose=False)


Copying annotations from: /home/asunkari/fnirs-representation-learning/snirf_dataset_2/Subj94/resting_hrf_20.snirf
Loading reference truth file: /home/asunkari/fnirs-representation-learning/snirf_dataset_2/Subj94/resting_hrf_20.snirf


/home/asunkari/fnirs-representation-learning/v1/fnirs_benchmark_v7_post_changes.py:475: RuntimeWarning: The data only contains 2D location information for the optode positions. It is highly recommended that data is used which contains 3D location information for the optode positions. With only 2D locations it can not be guaranteed that MNE functions will behave correctly and produce accurate results. If it is not possible to include 3D positions in your data, please consider using the set_montage() function.
  raw_source = mne.io.read_raw_snirf(annotation_source_file_path, preload=False, verbose=False)
/tmp/ipykernel_574312/389262916.py:22: RuntimeWarning: The data only contains 2D location information for the optode positions. It is highly recommended that data is used which contains 3D location information for the optode positions. With only 2D locations it can not be guaranteed that MNE functions will behave correctly and produce accurate results. If it is not possible to include 3D

Annotations: 34
True target pairs: ['S11_D25', 'S3_D2', 'S9_D17', 'S9_D23']
n true targets: 4
n true non-target pairs: 44


,subject,file_label,pair_name,sci_min,sci_mean,snr_min,snr_mean,negative_fraction_max,distance_m,group,midpoint_x,midpoint_y,midpoint_z,hemisphere
0,Subj94,no_hrf,S10_D17,0.371315,0.371315,3.314987,8.358619,0.0,0.030017,LS,-0.1425,-0.013,0.0,left
1,Subj94,no_hrf,S10_D18,0.299586,0.299586,3.619010,10.514975,0.0,0.030017,LS,-0.1575,-0.013,0.0,left
2,Subj94,no_hrf,S10_D21,0.189962,0.189962,2.611022,11.034301,0.0,0.008000,SS,-0.1500,0.004,0.0,left
3,Subj94,no_hrf,S10_D23,0.400225,0.400225,3.274005,17.924146,0.0,0.030017,LS,-0.1425,0.013,0.0,left
4,Subj94,no_hrf,S10_D24,0.231158,0.231158,2.781478,10.137124,0.0,0.030017,LS,-0.1575,0.013,0.0,left


In [7]:

# Quick QC sanity check.
pair_quality = state['pair_quality']
strict_bad = bench.get_bad_pair_names(pair_quality, 'strict_combined', config)
loose_bad = bench.get_bad_pair_names(pair_quality, 'loose_sci', config)

print('Strict bad pairs:', len(strict_bad))
print('Loose bad pairs:', len(loose_bad))
print('Strict good LS pairs:', int((pair_quality['group'].eq('LS') & ~pair_quality['pair_name'].isin(strict_bad)).sum()))
print('Loose good LS pairs:', int((pair_quality['group'].eq('LS') & ~pair_quality['pair_name'].isin(loose_bad)).sum()))


Strict bad pairs: 49
Loose bad pairs: 35
Strict good LS pairs: 6
Loose good LS pairs: 18


## Probe 1: Python-only pipeline on `no_hrf`

This tests whether the Python preprocessing + GLM path works when run serially in the notebook.


In [8]:

def run_python_probe(pipeline_label, file_label='no_hrf'):
    local_state = prepare_job_state(file_label)
    file_spec = local_state['file_spec']
    pipeline = next(ps for ps in config.pipeline_specs if ps.label == pipeline_label)
    raw_hb, bad_pairs = bench.preprocess_raw_to_hb(local_state['raw_cw'], local_state['pair_quality'], pipeline, config)
    print('Pipeline:', pipeline.label)
    print('Bad pairs:', len(bad_pairs))
    print('Available long channels after preprocess:', len(bench.get_available_long_channel_names(raw_hb, config)))
    t0 = time.time()
    result = bench.execute_python_pipeline(
        subject=SUBJECT,
        file_spec=file_spec,
        pipeline=pipeline,
        raw_cw=local_state['raw_cw'],
        raw_hb=raw_hb,
        target_pair_names=local_state['target_pair_set'],
        truth_templates=local_state['truth_templates'],
        snirf_file_path=local_state['snirf_file_path'],
        config=config,
    )
    dt = time.time() - t0
    print(f'Finished in {dt:.2f} s')
    print({k: len(v) if hasattr(v, '__len__') else type(v).__name__ for k, v in result.items()})
    return local_state, pipeline, raw_hb, result


In [9]:

py_state, py_pipeline, py_raw_hb, py_result = run_python_probe('LocalSS_Glover_AUTO', 'no_hrf')


Loading raw CW from: /home/asunkari/fnirs-representation-learning/snirf_dataset_2/Subj94/resting_clean.snirf
Copying annotations from: /home/asunkari/fnirs-representation-learning/snirf_dataset_2/Subj94/resting_hrf_20.snirf


/tmp/ipykernel_574312/389262916.py:15: RuntimeWarning: The data only contains 2D location information for the optode positions. It is highly recommended that data is used which contains 3D location information for the optode positions. With only 2D locations it can not be guaranteed that MNE functions will behave correctly and produce accurate results. If it is not possible to include 3D positions in your data, please consider using the set_montage() function.
  raw_cw = mne.io.read_raw_snirf(snirf_file_path, preload=True, verbose=False)
/home/asunkari/fnirs-representation-learning/v1/fnirs_benchmark_v7_post_changes.py:475: RuntimeWarning: The data only contains 2D location information for the optode positions. It is highly recommended that data is used which contains 3D location information for the optode positions. With only 2D locations it can not be guaranteed that MNE functions will behave correctly and produce accurate results. If it is not possible to include 3D positions in you

Loading reference truth file: /home/asunkari/fnirs-representation-learning/snirf_dataset_2/Subj94/resting_hrf_20.snirf


/tmp/ipykernel_574312/389262916.py:22: RuntimeWarning: The data only contains 2D location information for the optode positions. It is highly recommended that data is used which contains 3D location information for the optode positions. With only 2D locations it can not be guaranteed that MNE functions will behave correctly and produce accurate results. If it is not possible to include 3D positions in your data, please consider using the set_montage() function.
  reference_raw = mne.io.read_raw_snirf(reference_path, preload=True, verbose=False)


Pipeline: LocalSS_Glover_AUTO
Bad pairs: 49
Available long channels after preprocess: 12
Finished in 108.59 s
{'canonical_channel_metrics': 12, 'block_average_channel_metrics': 0, 'fir_channel_metrics': 0, 'shape_metrics': 12, 'roi_timecourses': 7004, 'nuisance_detail': 12, 'matlab_input_specs_list': 0, 'matlab_shift_specs_list': 0}


In [10]:
for key in ['canonical_channel_metrics', 'shape_metrics', 'roi_timecourses', 'nuisance_detail']:
    obj = py_result[key]
    print(f"\n=== {key} ===")
    if isinstance(obj, pd.DataFrame):
        display(obj.head())
    else:
        print(type(obj), len(obj) if hasattr(obj, "__len__") else "")


=== canonical_channel_metrics ===


,subject,file_label,amplitude_value,pipeline_label,backend,hrf_model,solver,channel_name,pair_name,chromophore,target_status,task_regressor,beta,nuisance_method_used,nuisance_regressor_label,nearest_short_distance_m,n_short_channels_used,t_value,p_value
0,Subj94,no_hrf,0,LocalSS_Glover_AUTO,python,glover,auto,S13_D26 hbo,S13_D26,hbo,true_non_target,task,-0.000009,pooled_fallback_average,pooled_fallback_average,0.271184,1,-2.682773,0.016338
1,Subj94,no_hrf,0,LocalSS_Glover_AUTO,python,glover,auto,S13_D26 hbr,S13_D26,hbr,true_non_target,task,0.000005,pooled_fallback_average,pooled_fallback_average,0.271184,1,0.915158,0.373696
2,Subj94,no_hrf,0,LocalSS_Glover_AUTO,python,glover,auto,S14_D31 hbo,S14_D31,hbo,true_non_target,task,0.000006,pooled_fallback_average,pooled_fallback_average,0.309631,1,3.246453,0.005059
3,Subj94,no_hrf,0,LocalSS_Glover_AUTO,python,glover,auto,S14_D31 hbr,S14_D31,hbr,true_non_target,task,-0.000008,pooled_fallback_average,pooled_fallback_average,0.309631,1,-5.138863,0.000099
4,Subj94,no_hrf,0,LocalSS_Glover_AUTO,python,glover,auto,S5_D4 hbo,S5_D4,hbo,true_non_target,task,0.000006,pooled_fallback_average,pooled_fallback_average,0.024418,1,0.760855,0.457811



=== shape_metrics ===


,subject,file_label,amplitude_value,pipeline_label,backend,channel_name,pair_name,chromophore,target_status,shape_source,...,curve_rmse,curve_nrmse,peak_latency_error_s,peak_amplitude_bias,peak_amplitude_ratio,auc_bias,recovered_peak_amplitude,recovered_auc,truth_peak_amplitude,truth_auc
0,Subj94,no_hrf,0,LocalSS_Glover_AUTO,python,S13_D26 hbo,S13_D26,hbo,true_non_target,beta_scaled_glover,...,9.530089e-07,NaN,17.80,7.717892e-07,NaN,-0.000009,7.717892e-07,-0.000009,0.0,0.0
1,Subj94,no_hrf,0,LocalSS_Glover_AUTO,python,S13_D26 hbr,S13_D26,hbr,true_non_target,beta_scaled_glover,...,5.409767e-07,NaN,17.80,-4.381071e-07,NaN,0.000005,-4.381071e-07,0.000005,0.0,0.0
2,Subj94,no_hrf,0,LocalSS_Glover_AUTO,python,S14_D31 hbo,S14_D31,hbo,true_non_target,beta_scaled_glover,...,6.286206e-07,NaN,10.52,1.937740e-06,NaN,0.000006,1.937740e-06,0.000006,0.0,0.0
3,Subj94,no_hrf,0,LocalSS_Glover_AUTO,python,S14_D31 hbr,S14_D31,hbr,true_non_target,beta_scaled_glover,...,9.466180e-07,NaN,10.52,-2.917975e-06,NaN,-0.000008,-2.917975e-06,-0.000008,0.0,0.0
4,Subj94,no_hrf,0,LocalSS_Glover_AUTO,python,S5_D4 hbo,S5_D4,hbo,true_non_target,beta_scaled_glover,...,6.219546e-07,NaN,10.52,1.917192e-06,NaN,0.000006,1.917192e-06,0.000006,0.0,0.0



=== roi_timecourses ===


,subject,file_label,amplitude_value,pipeline_label,backend,chromophore,target_status,curve_source,time_s,signal
0,Subj94,no_hrf,0,LocalSS_Glover_AUTO,python,hbo,true_non_target,roi_mean_beta_scaled_glover,-5.00,-0.0
1,Subj94,no_hrf,0,LocalSS_Glover_AUTO,python,hbo,true_non_target,roi_mean_beta_scaled_glover,-4.98,-0.0
2,Subj94,no_hrf,0,LocalSS_Glover_AUTO,python,hbo,true_non_target,roi_mean_beta_scaled_glover,-4.96,-0.0
3,Subj94,no_hrf,0,LocalSS_Glover_AUTO,python,hbo,true_non_target,roi_mean_beta_scaled_glover,-4.94,-0.0
4,Subj94,no_hrf,0,LocalSS_Glover_AUTO,python,hbo,true_non_target,roi_mean_beta_scaled_glover,-4.92,-0.0



=== nuisance_detail ===


,subject,file_label,pipeline_label,channel_name,chromophore,nuisance_method_used,nuisance_regressor_label,nearest_short_distance_m,n_short_channels_used
0,Subj94,no_hrf,LocalSS_Glover_AUTO,S13_D26 hbo,hbo,pooled_fallback_average,pooled_fallback_average,0.271184,1
1,Subj94,no_hrf,LocalSS_Glover_AUTO,S13_D26 hbr,hbr,pooled_fallback_average,pooled_fallback_average,0.271184,1
2,Subj94,no_hrf,LocalSS_Glover_AUTO,S14_D31 hbo,hbo,pooled_fallback_average,pooled_fallback_average,0.309631,1
3,Subj94,no_hrf,LocalSS_Glover_AUTO,S14_D31 hbr,hbr,pooled_fallback_average,pooled_fallback_average,0.309631,1
4,Subj94,no_hrf,LocalSS_Glover_AUTO,S5_D4 hbo,hbo,pooled_fallback_average,pooled_fallback_average,0.024418,1


## Probe 2: MATLAB bundle creation on `no_hrf`

This does **not** launch MATLAB yet. It just verifies that the AR-IRLS branch can build its design/spec bundle in Python.


In [11]:

mat_state, mat_pipeline, mat_raw_hb, mat_result = run_python_probe('LocalSS_Glover_ARIRLS', 'no_hrf')
print('Observed MATLAB specs:', len(mat_result['matlab_input_specs_list']))
print('Shift MATLAB specs:', len(mat_result['matlab_shift_specs_list']))


Loading raw CW from: /home/asunkari/fnirs-representation-learning/snirf_dataset_2/Subj94/resting_clean.snirf
Copying annotations from: /home/asunkari/fnirs-representation-learning/snirf_dataset_2/Subj94/resting_hrf_20.snirf


/tmp/ipykernel_574312/389262916.py:15: RuntimeWarning: The data only contains 2D location information for the optode positions. It is highly recommended that data is used which contains 3D location information for the optode positions. With only 2D locations it can not be guaranteed that MNE functions will behave correctly and produce accurate results. If it is not possible to include 3D positions in your data, please consider using the set_montage() function.
  raw_cw = mne.io.read_raw_snirf(snirf_file_path, preload=True, verbose=False)
/home/asunkari/fnirs-representation-learning/v1/fnirs_benchmark_v7_post_changes.py:475: RuntimeWarning: The data only contains 2D location information for the optode positions. It is highly recommended that data is used which contains 3D location information for the optode positions. With only 2D locations it can not be guaranteed that MNE functions will behave correctly and produce accurate results. If it is not possible to include 3D positions in you

Loading reference truth file: /home/asunkari/fnirs-representation-learning/snirf_dataset_2/Subj94/resting_hrf_20.snirf


/tmp/ipykernel_574312/389262916.py:22: RuntimeWarning: The data only contains 2D location information for the optode positions. It is highly recommended that data is used which contains 3D location information for the optode positions. With only 2D locations it can not be guaranteed that MNE functions will behave correctly and produce accurate results. If it is not possible to include 3D positions in your data, please consider using the set_montage() function.
  reference_raw = mne.io.read_raw_snirf(reference_path, preload=True, verbose=False)


Pipeline: LocalSS_Glover_ARIRLS
Bad pairs: 49
Available long channels after preprocess: 12
Finished in 233.38 s
{'canonical_channel_metrics': 0, 'block_average_channel_metrics': 0, 'fir_channel_metrics': 0, 'shape_metrics': 0, 'roi_timecourses': 0, 'nuisance_detail': 12, 'matlab_input_specs_list': 1, 'matlab_shift_specs_list': 1}
Observed MATLAB specs: 1
Shift MATLAB specs: 1


In [12]:

job_dir = config.output_path() / 'notebook_debug' / SUBJECT / 'no_hrf'
job_dir.mkdir(parents=True, exist_ok=True)

bundle_path = bench.write_matlab_bundle(
    mat_result['matlab_input_specs_list'],
    mat_result['matlab_shift_specs_list'],
    job_dir,
)
print('Bundle path:', bundle_path)
print(bundle_path.read_text()[:1000] if bundle_path else 'No bundle written')


Bundle path: /home/asunkari/fnirs-representation-learning/outputs_benchmark_v7/notebook_debug/Subj94/no_hrf/matlab_inputs/matlab_bundle.json
{
  "input_mat_files": [
    "/home/asunkari/fnirs-representation-learning/outputs_benchmark_v7/notebook_debug/Subj94/no_hrf/matlab_inputs/0001__LocalSS_Glover_ARIRLS__observed__input.mat",
    "/home/asunkari/fnirs-representation-learning/outputs_benchmark_v7/notebook_debug/Subj94/no_hrf/matlab_inputs/0002__LocalSS_Glover_ARIRLS__shift001__input.mat"
  ],
  "output_csv_files": [
    "/home/asunkari/fnirs-representation-learning/outputs_benchmark_v7/notebook_debug/Subj94/no_hrf/matlab_inputs/0001__LocalSS_Glover_ARIRLS__observed__output.csv",
    "/home/asunkari/fnirs-representation-learning/outputs_benchmark_v7/notebook_debug/Subj94/no_hrf/matlab_inputs/0002__LocalSS_Glover_ARIRLS__shift001__output.csv"
  ]
}


## Probe 3: MATLAB Engine run in the notebook main process

This is the crucial test. If this works here but the batch script still dies, the likely issue is **MATLAB Engine inside the worker process**, not the math itself.


In [13]:
import matlab.engine
eng = matlab.engine.start_matlab("-nojvm")
print(eng.version())

25.2.0.3150157 (R2025b) Update 4


In [14]:
helper_dir = str(SCRIPT_PATH.parent)
eng.addpath(helper_dir, nargout=0)

eng.setenv("FNIRS_BUNDLE_JSON", str(bundle_path), nargout=0)
eng.setenv("FNIRS_ANALYZIR_PATH", str(config.analyzir_path), nargout=0)

print(eng.eval("which('nirs.modules.AR_IRLS')", nargout=1))
print(eng.eval("which('nirs.math.ar_irls')", nargout=1))
print(eng.eval(f"which('{Path(helper_dir, 'analyzir_arirls_batch.m').stem}')", nargout=1))

/home/asunkari/nirs-toolbox/+nirs/+modules/AR_IRLS.m
/home/asunkari/nirs-toolbox/+nirs/+math/ar_irls.m
/home/asunkari/fnirs-representation-learning/v1/analyzir_arirls_batch.m


In [15]:
helper_m = SCRIPT_PATH.with_name("analyzir_arirls_batch.m")
print(helper_m)
print(helper_m.exists())

/home/asunkari/fnirs-representation-learning/v1/analyzir_arirls_batch.m
True


In [16]:
print(bench.run_matlab_sidecar_engine)

<function run_matlab_sidecar_engine at 0x758274318860>


In [ ]:
matlab_df = None
try:
    t0 = time.time()
    matlab_df = bench.run_matlab_sidecar_engine(config, bundle_path, job_dir, helper_m)
    dt = time.time() - t0
    print(f"run_matlab_sidecar_engine finished in {dt:.2f} s")
    display(matlab_df.head())
    print("Rows:", len(matlab_df))
except Exception as exc:
    print("run_matlab_sidecar_engine failed:")
    print(type(exc).__name__, exc)
    traceback.print_exc()

    for candidate in ["matlab_engine.log", "matlab_engine_fallback.log", "matlab_batch.log"]:
        p = job_dir / candidate
        if p.exists():
            print(f"\n--- {candidate} ---")
            print(p.read_text()[:4000])

.

In [ ]:
"""
matlab_df = None
try:
    t0 = time.time()
    matlab_df = bench.run_matlab_sidecar(config, bundle_path, job_dir)
    dt = time.time() - t0
    print(f"MATLAB sidecar finished in {dt:.2f} s")
    display(matlab_df.head())
    print("Rows:", len(matlab_df))
except Exception as exc:
    print("MATLAB sidecar failed:")
    print(type(exc).__name__, exc)
    traceback.print_exc()

    for candidate in ["matlab_engine.log", "matlab_engine_fallback.log", "matlab_batch.log"]:
        p = job_dir / candidate
        if p.exists():
            print(f"\n--- {candidate} ---")
            print(p.read_text()[:4000])
"""

.

## Optional: run the same representative Python probe on `hrf_20`

This lets you confirm the active-condition path separately from the null path.


In [ ]:

hrf20_state, hrf20_pipeline, hrf20_raw_hb, hrf20_result = run_python_probe('LocalSS_Glover_AUTO', 'hrf_20')


In [ ]:

if isinstance(hrf20_result['canonical_channel_metrics'], pd.DataFrame):
    display(hrf20_result['canonical_channel_metrics'].head())
if isinstance(hrf20_result['shape_metrics'], pd.DataFrame):
    display(hrf20_result['shape_metrics'].head())


## Interpretation guide

- If **Probe 1 fails**, the issue is in Python preprocessing / GLM before MATLAB.
- If **Probe 1 works but Probe 2 fails**, the issue is in AR-IRLS spec construction.
- If **Probe 2 works but Probe 3 fails**, the issue is MATLAB runtime/path/helper behavior.
- If **Probe 3 works in the notebook but the batch script still dies**, the likely culprit is the **worker-process execution path**, especially MATLAB Engine inside the `ProcessPoolExecutor` worker.
